In [1]:
import numpy as np
import pandas as pd

In [2]:
path = r"C:\Users\JOKER\Desktop\berlin\Fächer\Summer School\Power Markets\Data"
actual = pd.read_csv(f"{path}/actual_portfolio_load.csv")
load = pd.read_csv(f"{path}/forecast_load.csv")  #from task1
imb = pd.read_csv(f"{path}/imbalance_prices.csv")
da = pd.read_csv(f"{path}/day_ahead_prices.csv")

In [3]:
grid = actual[["timestamp_utc", "total_actual_mwh"]].merge(
    load[["timestamp_utc", "total_forecast_mwh"]], on="timestamp_utc").merge(
    imb[["timestamp_utc", "imbalance_price_eur_mwh"]], on="timestamp_utc").merge(
    da[["timestamp_utc", "day_ahead_price_eur_mwh"]], on="timestamp_utc")
A = grid["total_actual_mwh"].to_numpy()
L = grid["total_forecast_mwh"].to_numpy()
P_imb = grid["imbalance_price_eur_mwh"].to_numpy()
P_da = grid["day_ahead_price_eur_mwh"].to_numpy()

In [4]:
imbalance_volume = A - L
c_imb = imbalance_volume @ P_imb
premium_imb = imbalance_volume @ (P_imb - P_da)  #penalty/cost of being inaccurate

In [5]:
print("total imbalance volume:", round(imbalance_volume.sum(), 2), "MWh")
print("positive imbalance (consumed more than scheduled):", round(imbalance_volume[imbalance_volume > 0].sum(), 2), "MWh")
print("negative imbalance (consumed less than scheduled):", round(imbalance_volume[imbalance_volume < 0].sum(), 2), "MWh")
print("C_imb:", round(c_imb, 2), "EUR")
print("Premium_imb:", round(premium_imb, 2), "EUR")

total imbalance volume: 138.0 MWh
positive imbalance (consumed more than scheduled): 491.98 MWh
negative imbalance (consumed less than scheduled): -353.98 MWh
C_imb: 30503.84 EUR
Premium_imb: 13589.24 EUR


In [6]:
for name, tag in [("UNHEDGED", "None"), ("Coarse hedge (CAL)", "coarse"), ("Granular hedge (M01-M03, Q2-Q4)", "granular")]:
    residual = pd.read_csv(f"{path}/residual_" + tag + ".csv")["day_ahead_residual"].to_numpy()  #from task4
    c_residual_da = residual @ P_da
    if tag == "None":
        c_futures = 0.0
    else:
        pos = pd.read_csv(f"{path}/task3_positions_" + tag + ".csv")
        c_futures = (pos["hedge_energy"] * pos["price_eur_mwh"]).sum()
    print("\n", name)
    print("  C_futures:", round(c_futures, 2), "EUR")
    print("  C_residual_DA:", round(c_residual_da, 2), "EUR")
    print("  C_imb:", round(c_imb, 2), "EUR")
    print("  C_final:", round(c_futures + c_residual_da + c_imb, 2), "EUR")


 UNHEDGED
  C_futures: 0.0 EUR
  C_residual_DA: 3506703.99 EUR
  C_imb: 30503.84 EUR
  C_final: 3537207.83 EUR

 Coarse hedge (CAL)
  C_futures: 3477391.03 EUR
  C_residual_DA: -26162.16 EUR
  C_imb: 30503.84 EUR
  C_final: 3481732.71 EUR

 Granular hedge (M01-M03, Q2-Q4)
  C_futures: 3477395.26 EUR
  C_residual_DA: -25681.97 EUR
  C_imb: 30503.84 EUR
  C_final: 3482217.13 EUR


In [7]:
pd.DataFrame({"timestamp_utc": grid["timestamp_utc"], "imbalance_volume": imbalance_volume, "imbalance_cashflow_eur": imbalance_volume * P_imb}).to_csv(f"{path}/imbalance.csv", index=False)